# Digital Pathology Assistant — Colab (one file)

**Upload only this notebook.** No other project files needed.

1. `Runtime → Change runtime type → GPU`
2. `Runtime → Run all`

The notebook downloads **real public pathology tiles** (PatchCamelyon from Zenodo),
trains a classifier on GPU, shows metrics and plots, and lets you download a
CPU-loadable checkpoint.

> Decision-support demo only — not a clinical device.

## 1. Setup

In [ ]:
# Light deps. Colab already has torch + CUDA.
!pip install -q matplotlib

In [ ]:
import os, time, urllib.request
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Torch', torch.__version__, '| device:', DEVICE)
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

## 2. Settings (edit these)

In [ ]:
# --- you control these ---
MAX_SAMPLES = 8000      # subsample for speed (full valid set = 32768)
EPOCHS      = 10
BATCH_SIZE  = 64
LR          = 1e-3
VAL_FRACTION = 0.2
SEED        = 42
TILE_SIZE   = 96        # PatchCamelyon tiles are 96x96
# -------------------------

DATA_DIR = Path('data/pcam')
MODEL_PATH = Path('tile_classifier.pt')
DATA_DIR.mkdir(parents=True, exist_ok=True)

## 3. Download public data (PatchCamelyon)

Source: [Zenodo — PatchCamelyon](https://zenodo.org/record/2546921)  
96×96 H&E tiles from lymph-node slides. Label 0 = normal tissue, 1 = metastasis.

We use the **validation split** (~300 MB) so Colab can download it quickly.
For a full study, swap the URLs to the `train` files on the same Zenodo page.

In [ ]:
ZENODO = 'https://zenodo.org/record/2546921/files'
FILES = {
    'camelyonpatch_level_2_split_valid_x.npz': DATA_DIR / 'valid_x.npz',
    'camelyonpatch_level_2_split_valid_y.npz': DATA_DIR / 'valid_y.npz',
}

def download(url, dest):
    if dest.exists():
        print('already have', dest.name)
        return
    print('downloading', dest.name, '...')
    urllib.request.urlretrieve(url, dest)
    print('saved', dest, f'({dest.stat().st_size / 1e6:.1f} MB)')

for remote, local in FILES.items():
    download(f'{ZENODO}/{remote}', local)

In [ ]:
def load_pcam(x_path, y_path, max_samples, seed):
    """Load PatchCamelyon tiles -> (N,96,96,3) uint8, (N,) float labels."""
    x = np.load(x_path)['x']          # (N, 96, 96, 3)
    y = np.load(y_path)['y']          # (N, 1, 1, 1)
    labels = y.reshape(-1).astype(np.float32)
    n = min(max_samples, len(x))
    rng = np.random.default_rng(seed)
    idx = rng.choice(len(x), size=n, replace=False)
    return x[idx], labels[idx]

tiles, labels = load_pcam(DATA_DIR / 'valid_x.npz', DATA_DIR / 'valid_y.npz',
                          MAX_SAMPLES, SEED)
print(f'Loaded {len(tiles)} tiles | metastasis: {int(labels.sum())} | normal: {int((labels==0).sum())}')

In [ ]:
# Look at a few real tiles
fig, axes = plt.subplots(2, 4, figsize=(10, 5))
for row, (lab, name) in enumerate([(0, 'normal'), (1, 'metastasis')]):
    idx = np.where(labels == lab)[0][:4]
    for col, i in enumerate(idx):
        axes[row, col].imshow(tiles[i])
        axes[row, col].axis('off')
        if col == 0:
            axes[row, col].set_title(name)
plt.suptitle('PatchCamelyon samples'); plt.tight_layout(); plt.show()

## 4. Model + training code (all in this notebook)

In [ ]:
PIXEL_SCALE, NORM_MEAN, NORM_STD = 255.0, 0.5, 0.5

def tiles_to_tensor(batch_np):
    """(B,H,W,3) uint8 -> (B,3,H,W) float, same norm for train & inference."""
    x = batch_np.astype(np.float32) / PIXEL_SCALE
    x = (x - NORM_MEAN) / NORM_STD
    return torch.from_numpy(x).permute(0, 3, 1, 2)


class TileClassifier(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 16, 3, padding=1), nn.BatchNorm2d(16), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(16, 32, 3, padding=1), nn.BatchNorm2d(32), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(),
            nn.AdaptiveAvgPool2d(1),
        )
        self.head = nn.Linear(64, 1)

    def forward(self, x):
        return self.head(self.features(x).flatten(1)).squeeze(1)


def split_data(tiles, labels, val_fraction, seed):
    rng = np.random.default_rng(seed)
    order = rng.permutation(len(tiles))
    n_val = max(1, int(len(tiles) * val_fraction))
    val_idx, train_idx = order[:n_val], order[n_val:]
    return (tiles[train_idx], labels[train_idx]), (tiles[val_idx], labels[val_idx])


def iter_batches(x, y, batch_size, shuffle):
    idx = np.arange(len(x))
    if shuffle:
        np.random.shuffle(idx)
    for start in range(0, len(idx), batch_size):
        b = idx[start:start + batch_size]
        yield x[b], y[b]


@torch.no_grad()
def evaluate(model, val_x, val_y, batch_size, device, loss_fn):
    model.eval()
    probs, true, total_loss, n = [], [], 0.0, 0
    for bx, by in iter_batches(val_x, val_y, batch_size, False):
        inp = tiles_to_tensor(bx).to(device)
        tgt = torch.from_numpy(by).to(device)
        logits = model(inp)
        total_loss += loss_fn(logits, tgt).item()
        probs.append(torch.sigmoid(logits).cpu().numpy())
        true.append(by)
        n += 1
    probs = np.concatenate(probs)
    true = np.concatenate(true)
    acc = ((probs >= 0.5).astype(np.float32) == true).mean()
    return true, probs, acc, total_loss / max(1, n)

## 5. Train

In [ ]:
(train_x, train_y), (val_x, val_y) = split_data(tiles, labels, VAL_FRACTION, SEED)
print(f'train {len(train_x)} | val {len(val_x)}')

model = TileClassifier().to(DEVICE)
opt = torch.optim.Adam(model.parameters(), lr=LR)
loss_fn = nn.BCEWithLogitsLoss()
history = []

for epoch in range(1, EPOCHS + 1):
    model.train()
    running, batches, t0 = 0.0, 0, time.time()
    for bx, by in iter_batches(train_x, train_y, BATCH_SIZE, shuffle=True):
        inp = tiles_to_tensor(bx).to(DEVICE)
        tgt = torch.from_numpy(by).to(DEVICE)
        opt.zero_grad()
        loss = loss_fn(model(inp), tgt)
        loss.backward()
        opt.step()
        running += loss.item()
        batches += 1
    val_true, val_prob, val_acc, val_loss = evaluate(model, val_x, val_y, BATCH_SIZE, DEVICE, loss_fn)
    train_loss = running / max(1, batches)
    history.append(dict(epoch=epoch, train_loss=train_loss, val_loss=val_loss, val_acc=val_acc))
    print(f'epoch {epoch:>2}/{EPOCHS}  train {train_loss:.4f}  val {val_loss:.4f}  acc {val_acc:.3f}  ({time.time()-t0:.1f}s)')

## 6. Understand the results

In [ ]:
ep = [h['epoch'] for h in history]
fig, (a1, a2) = plt.subplots(1, 2, figsize=(11, 4))
a1.plot(ep, [h['train_loss'] for h in history], 'o-', label='train')
a1.plot(ep, [h['val_loss'] for h in history], 'o-', label='val')
a1.set_title('Loss'); a1.legend(); a1.grid(alpha=.3)
a2.plot(ep, [h['val_acc'] for h in history], 'o-', color='green')
a2.set_title('Validation accuracy'); a2.set_ylim(0, 1); a2.grid(alpha=.3)
plt.tight_layout(); plt.show()

In [ ]:
y_true = val_true.astype(int)
y_pred = (val_prob >= 0.5).astype(int)
tp = int(((y_pred==1)&(y_true==1)).sum())
tn = int(((y_pred==0)&(y_true==0)).sum())
fp = int(((y_pred==1)&(y_true==0)).sum())
fn = int(((y_pred==0)&(y_true==1)).sum())
cm = np.array([[tn, fp], [fn, tp]])

prec = tp/(tp+fp) if tp+fp else 0
rec  = tp/(tp+fn) if tp+fn else 0
f1   = 2*prec*rec/(prec+rec) if prec+rec else 0

fig, ax = plt.subplots(figsize=(4.5, 4))
ax.imshow(cm, cmap='Blues')
for i in range(2):
    for j in range(2):
        ax.text(j, i, cm[i,j], ha='center', va='center', fontsize=14)
ax.set_xticks([0,1]); ax.set_xticklabels(['pred neg','pred pos'])
ax.set_yticks([0,1]); ax.set_yticklabels(['true neg','true pos'])
ax.set_title('Confusion matrix'); plt.show()
print(f'Precision {prec:.3f} | Recall (sensitivity) {rec:.3f} | F1 {f1:.3f}')
print('Recall = missed metastases when too low')

In [ ]:
plt.figure(figsize=(8,4))
plt.hist(val_prob[y_true==0], bins=25, alpha=.6, label='true normal')
plt.hist(val_prob[y_true==1], bins=25, alpha=.6, label='true metastasis')
plt.axvline(.5, color='k', ls='--', label='threshold')
plt.xlabel('predicted metastasis probability'); plt.legend(); plt.show()

In [ ]:
# Sample predictions you can eyeball
fig, axes = plt.subplots(2, 4, figsize=(10, 5))
for row, (lab, name) in enumerate([(0,'normal'), (1,'metastasis')]):
    idx = np.where(val_true == lab)[0][:4]
    for col, i in enumerate(idx):
        axes[row,col].imshow(val_x[i])
        axes[row,col].axis('off')
        axes[row,col].set_title(f'p={val_prob[i]:.2f}')
        if col==0: axes[row,col].set_ylabel(name)
plt.suptitle('Validation tiles + model score'); plt.tight_layout(); plt.show()

## 7. Triage demo (decision-support workflow)

Group tiles into a fake "case", rank by mean top-tile score, and show which
regions a pathologist should review first.

In [ ]:
@torch.no_grad()
def score_tiles(model, tile_array, device, batch_size=64):
  model.eval()
  out = []
  for start in range(0, len(tile_array), batch_size):
    batch = tile_array[start:start+batch_size]
    logits = model(tiles_to_tensor(batch).to(device))
    out.append(torch.sigmoid(logits).cpu().numpy())
  return np.concatenate(out)

# Treat 64 random val tiles as one "slide"
demo_tiles = val_x[:64]
demo_scores = score_tiles(model, demo_tiles, DEVICE)
case_score = demo_scores[np.argsort(demo_scores)[-10:]].mean()  # mean of top-10 tiles
priority = 'URGENT' if case_score >= .85 else ('HIGH' if case_score >= .6 else 'ROUTINE')
top_idx = np.argsort(demo_scores)[::-1][:6]

print(f'Case score {case_score:.2f} -> priority {priority}')
print('Top regions to review:')
for rank, i in enumerate(top_idx, 1):
    print(f'  {rank}. tile #{i}  score {demo_scores[i]:.2f}')

fig, axes = plt.subplots(1, 6, figsize=(12, 2.5))
for ax, i in zip(axes, top_idx):
    ax.imshow(demo_tiles[i]); ax.axis('off')
    ax.set_title(f'{demo_scores[i]:.2f}', color='red' if demo_scores[i]>=.5 else 'green')
plt.suptitle('Suggested review regions (highest scores)'); plt.tight_layout(); plt.show()

## 8. Save & download checkpoint (runs on CPU later)

In [ ]:
checkpoint = {
    'checkpoint_format': 1,
    'tile_size': TILE_SIZE,
    'version': 'colab-0.1',
    'dataset': 'PatchCamelyon (Zenodo 2546921)',
    'state_dict': {k: v.cpu() for k, v in model.state_dict().items()},
}
torch.save(checkpoint, MODEL_PATH)
print('Saved', MODEL_PATH.resolve())

In [ ]:
from google.colab import files
files.download(str(MODEL_PATH))

## 9. Reload on CPU (sanity check)

Proves the downloaded file works without a GPU.

In [ ]:
cpu_model = TileClassifier()
ckpt = torch.load(MODEL_PATH, map_location='cpu')
cpu_model.load_state_dict(ckpt['state_dict'])
cpu_model.eval()
test_prob = score_tiles(cpu_model, val_x[:8], 'cpu')[0]
print('CPU inference OK, first score:', round(float(test_prob), 3))